<a href="https://colab.research.google.com/github/chelseanbr/aih-final-project/blob/main/notebooks/02-generate-mcqs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning a Large Language Model for Layperson Medical Question Answering
AI in Healthcare | MSAI Spr '25 | Chelsea Ramos

## Objectives
1. Explore and sample the MedQuAD dataset
2. ***Generate ~3.5k medical multiple-choice questions (MCQs) using Phi-2***
3. Fine-tune TinyLLaMA with the generated MCQs
4. Evaluate model accuracy on held-out MCQs
5. Compare against other models (baseline and zero-shot)

---

# 2. Generate MCQs using Phi-2

In [2]:
import json
import re
from pathlib import Path
from tqdm import tqdm
from google.colab import files

# Upload JSONL file with question and answer fields
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

Saving medquad_sampled.jsonl to medquad_sampled.jsonl


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# checkpoint = "microsoft/phi-2"
# checkpoint = "stanford-crfm/BioMedLM"
checkpoint = "medalpaca/medalpaca-7b"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Load and parse JSONL file
with open(file_path, "r") as f:
    qa_data = [json.loads(line) for line in f]

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(checkpoint, torch_dtype=torch.float16).to(device)
model.eval()
print('Tokenizer and model loaded.')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/7.18G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:609: UserWarning: `pad_token_id` should be positive but got -1. This will cause errors when batch generating, if there is padding. Please set `pad_token_id` explicitly as `model.generation_config.pad_token_id=PAD_TOKEN_ID` to avoid errors in generation
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Tokenizer and model loaded.


In [5]:
# Check if output file already exists
output_file = Path("generated_mcqs.jsonl")
if output_file.exists():
    response = input(f"File {output_file} exists. Delete it? (y/n): ")
    if response.lower().startswith("y"):
        output_file.unlink()
    else:
        raise RuntimeError("Aborting to avoid overwriting the file.")

# Few-shot example to encourage structure
few_shot_example = """
<question>
What is asthma?
A. A skin condition
B. A viral infection
C. A chronic lung disease that causes airway inflammation
D. A type of heart failure
</question>

<response>
Asthma is a chronic lung condition that causes airway inflammation, which can make breathing difficult.
</response>

<answer>
C
</answer>
"""

# Format QA into prompt
def format_prompt(question, answer):
    return f"""
You are a helpful medical assistant. Your task is to convert a medical question and answer pair into a multiple-choice question with four options: one correct answer and three medically plausible but incorrect distractors.

Please return your output in this structured XML format:

<question>
[Multiple-choice question text, including labeled options A., B., C., D.]
</question>

<response>
[Concise explanation of why the correct answer is right, and optionally why others are wrong.]
</response>

<answer>
[Correct answer letter: A, B, C, or D]
</answer>

Here is an example:
{few_shot_example}

Now, use the following Q&A to create a new MCQ:

Question: {question}
Answer: {answer}
"""

# Batch-generate MCQs
# BATCH_SIZE = 8
BATCH_SIZE = 2
results = []
for i in tqdm(range(0, BATCH_SIZE, BATCH_SIZE),
              desc=f"LLM Running on Micro Batches {BATCH_SIZE}"):  # DEBUG
# for i in tqdm(range(0, len(qa_data), BATCH_SIZE,
#                     desc=f"LLM Running on Micro Batches {BATCH_SIZE}"):
    batch = qa_data[i:i + BATCH_SIZE]
    prompts = [format_prompt(q["question"], q["answer"]) for q in batch]
    # questions = [q["question"] for q in batch]

    inputs = tokenizer(prompts, return_tensors="pt",
                       truncation=True,
                       padding=True).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            # input_ids=inputs["input_ids"],
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            # attention_mask=inputs["attention_mask"]
        )

    prompt_lengths = [len(x) for x in inputs["input_ids"]]
    for j, output in enumerate(outputs):
        generated_tokens = output[prompt_lengths[j]:]  # remove prompt
        completion = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        results.append({
            # "question": batch[j]["question"],
            # "answer": batch[j]["answer"],
            "mcq": completion.strip()
        })


    # decoded = tokenizer.batch_decode(outputs[:, len(inputs["input_ids"][0]) :],
    #                                  skip_special_tokens=True)
    # # decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # # for question, generated in zip(questions, decoded):
    # #     results.append({"question": question, "answer": generated})
    # for mcqa in decoded:
    #     q_match = re.search(r"<question>(.*?)</question>", mcqa)
    #     question = q_match.group(1) if q_match else None
    #     r_match = re.search(r"<response>(.*?)</response>", mcqa)
    #     response = r_match.group(1) if r_match else None
    #     a_match = re.search(r"<answer>(.*?)</answer>", mcqa)
    #     answer = a_match.group(1) if a_match else None
    #     if question and response and answer:
    #         results.append({"question": question, "response": response,
    #                         "answer": answer, "full_text": mcqa})

# Save to JSONL file
with open(output_file, "w") as f:
    for entry in results:
        f.write(json.dumps(entry) + "\n")

print(f"Generated MCQs saved to {output_file}")

File generated_mcqs.jsonl exists. Delete it? (y/n): y


LLM Running on Micro Batches 2: 100%|██████████| 1/1 [00:06<00:00,  6.83s/it]

Generated MCQs saved to generated_mcqs.jsonl
